# AutoBio: A Simulation and Benchmark for Robotic Automation in Digital Biology Laboratory

**A Self-Contained Tutorial Notebook**

---

**Paper:** Lan et al., 2025 — *AutoBio: A Simulation and Benchmark for Robotic Automation in Digital Biology Laboratory* (arXiv:2505.14030)  
**Repository:** https://github.com/autobio-bench/AutoBio

---

## What You Will Learn

This notebook walks you through the key ideas behind AutoBio from the ground up:

1. **Motivation** — Why robotic automation in biology labs is hard
2. **Architecture Overview** — How the AutoBio framework is structured
3. **Physics Plugins** — Thread, Detent, and Eccentric mechanisms; quasi-static liquid
4. **Benchmark Tasks** — Three difficulty tiers with biological primitives
5. **Evaluation Metrics** — How VLA model performance is measured
6. **Hands-On Simulations** — Visualizing core concepts with Python code
7. **Baseline Results & Analysis** — Understanding where current models fall short
8. **Exercises** — Deepen your understanding with guided challenges

---

> **Prerequisites:** Basic Python, NumPy, and Matplotlib familiarity. No robotics background required!

## Section 0 — Environment Setup

In [ ]:
# Install required packages (lightweight — no simulator required for this tutorial)
# !pip install numpy matplotlib scipy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch, Circle, FancyBboxPatch, Arc
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial.transform import Rotation
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
print("✅ All packages loaded successfully!")

---
## Section 1 — Motivation: Why Are Biology Labs Hard for Robots?

Modern biology labs are dominated by repetitive, time-consuming manual tasks—pipetting, centrifugation, plate reading, and reagent mixing. These are ideal targets for automation. However, **general-purpose robotic systems** built for home or factory settings struggle badly in labs because of three core challenges:

| Challenge | Example | Why It's Hard |
|---|---|---|
| **Precision manipulation** | Aligning a tube into a centrifuge slot (< 1 mm tolerance) | Current VLA models trained on coarse pick-and-place fail |
| **Transparent materials** | Glass beakers, liquid samples, clear tubes | Standard rendering doesn't simulate refraction/transparency well |
| **Digital interfaces** | LCD displays on centrifuges, PCR machines | Robots must *read* and *interact with* on-device screens |
| **Long-horizon protocols** | A PCR workflow spans 10+ sequential sub-tasks | Errors compound; recovery is critical |

**AutoBio** (Lan et al., 2025) is the first benchmark designed specifically to close this gap.

In [ ]:
# ─── Figure 1-A: Visualising the precision requirement in lab tasks ───────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Figure 1 — Why Lab Automation Is Uniquely Challenging",
             fontsize=14, fontweight='bold', y=1.02)

# --- Panel A: Tolerance comparison across domains ---
ax = axes[0]
tasks   = ['Pick & Place\n(factory)', 'Cloth Folding\n(home)', 'Tube Insertion\n(lab)', 'Pipette Tip\n(lab)']
tol_mm  = [10, 20, 0.8, 0.3]
colors  = ['#4CAF50', '#2196F3', '#FF5722', '#E91E63']
bars = ax.barh(tasks, tol_mm, color=colors, edgecolor='black', linewidth=0.8)
ax.set_xlabel('Allowed Position Error (mm)', fontsize=10)
ax.set_title('A) Precision Requirements\nAcross Task Domains', fontsize=11)
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=1.5, label='1 mm threshold')
ax.legend(fontsize=8)
for bar, val in zip(bars, tol_mm):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val} mm', va='center', fontsize=9)

# --- Panel B: Task length distribution ---
ax = axes[1]
categories = ['Meta-World\n(factory)', 'LIBERO\n(home)', 'AutoBio\n(lab)']
avg_steps  = [5, 8, 18]
colors2    = ['#4CAF50', '#2196F3', '#E91E63']
ax.bar(categories, avg_steps, color=colors2, edgecolor='black', linewidth=0.8, width=0.5)
ax.set_ylabel('Average Steps per Task', fontsize=10)
ax.set_title('B) Task Horizon\nComparison', fontsize=11)
for i, (cat, val) in enumerate(zip(categories, avg_steps)):
    ax.text(i, val + 0.3, str(val), ha='center', fontsize=11, fontweight='bold')

# --- Panel C: Radar chart of challenge dimensions ---
ax = axes[2]
ax.set_aspect('equal')
dimensions = ['Precision\nManipulation', 'Visual\nReasoning', 'Long\nHorizon',
              'Multimodal\nInteraction', 'Transparent\nMaterials']
N = len(dimensions)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close polygon

home_scores = [2, 3, 3, 2, 1]
lab_scores  = [5, 5, 5, 5, 5]
home_scores += home_scores[:1]
lab_scores  += lab_scores[:1]

ax2 = fig.add_subplot(133, polar=True)
axes[2].set_visible(False)  # hide the cartesian placeholder
ax2.set_position(axes[2].get_position())
ax2.plot(angles, home_scores, 'o-', color='#2196F3', linewidth=2, label='Home Tasks')
ax2.fill(angles, home_scores, alpha=0.2, color='#2196F3')
ax2.plot(angles, lab_scores,  'o-', color='#E91E63', linewidth=2, label='Lab Tasks (AutoBio)')
ax2.fill(angles, lab_scores,  alpha=0.2, color='#E91E63')
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(dimensions, fontsize=8)
ax2.set_ylim(0, 5)
ax2.set_yticks([1,2,3,4,5])
ax2.set_yticklabels([])
ax2.set_title('C) Challenge Profile\n', fontsize=11, pad=15)
ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)

plt.tight_layout()
plt.savefig('section1_motivation.png', bbox_inches='tight')
plt.show()
print("Observation: Lab tasks demand much higher precision and longer horizons than domestic benchmarks.")

---
## Section 2 — AutoBio Framework Architecture

AutoBio is built on top of **MuJoCo** and organises the world into three layers:

```
┌─────────────────────────────────────────────────────────┐
│                   AutoBio Framework                     │
│                                                         │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐  │
│  │  3-D Assets  │  │   Physics    │  │  Rendering   │  │
│  │  Pipeline    │  │   Plugins    │  │  Stack       │  │
│  │              │  │              │  │              │  │
│  │ • Scan real  │  │ • Thread     │  │ • PBR        │  │
│  │   instruments│  │ • Detent     │  │ • Transparent│  │
│  │ • Mesh clean │  │ • Eccentric  │  │   materials  │  │
│  │ • URDF gen   │  │ • Quasi-     │  │ • Dynamic UI │  │
│  │              │  │   static liq.│  │   rendering  │  │
│  └──────────────┘  └──────────────┘  └──────────────┘  │
│                          │                              │
│              ┌──────────────────────┐                  │
│              │   Benchmark Layer    │                  │
│              │ Easy / Medium / Hard │                  │
│              │ (Biological Tasks)   │                  │
│              └──────────────────────┘                  │
│                          │                              │
│              ┌──────────────────────┐                  │
│              │   VLA Model API      │                  │
│              │ OpenVLA / RDT-1B     │                  │
│              └──────────────────────┘                  │
└─────────────────────────────────────────────────────────┘
```

### Key design principles
- **Biological primitives first:** Tasks are decomposed into fundamental actions like *pipette*, *centrifuge*, *open cap*, *align slot*.
- **Instrument digitization pipeline:** Real lab instruments are 3-D scanned, cleaned, and converted to simulation-ready MJCF assets.
- **Custom physics plugins** extend MuJoCo to handle mechanisms that standard rigid-body engines cannot model (detailed in Section 3).
- **Physically-based rendering (PBR)** supports transparent glass and liquid, making visual observations photorealistic.
- **Dynamic UI:** LCD displays on centrifuges and PCR machines render live state.

In [ ]:
# ─── Figure 2: AutoBio pipeline block diagram ─────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('Figure 2 — AutoBio End-to-End Pipeline', fontsize=14, fontweight='bold')

def draw_box(ax, x, y, w, h, title, items, color):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='#333', linewidth=1.5, alpha=0.85)
    ax.add_patch(box)
    ax.text(x + w/2, y + h - 0.35, title, ha='center', va='top',
            fontsize=10, fontweight='bold', color='#111')
    for i, item in enumerate(items):
        ax.text(x + w/2, y + h - 0.78 - i*0.38, item, ha='center', va='top',
                fontsize=8.5, color='#222')

def draw_arrow(ax, x1, x2, y):
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))

# Real-world instruments
draw_box(ax, 0.3, 2.5, 2.8, 3.0, '① Real Lab\nInstruments',
         ['• Centrifuge', '• PCR Machine', '• Pipette', '• Microscope', '• Vortex Mixer'],
         '#FFF9C4')

# 3D scan → URDF
draw_box(ax, 3.5, 2.5, 2.8, 3.0, '② Asset Digitization',
         ['• 3-D scanning', '• Mesh cleaning', '• Collision proxy gen.', '• MJCF/URDF export', '• Texture baking'],
         '#C8E6C9')

# Physics plugins
draw_box(ax, 6.7, 2.5, 2.8, 3.0, '③ Physics Plugins',
         ['• Thread mechanism', '• Detent mechanism', '• Eccentric mechanism', '• Quasi-static liquid', '• Rigid contacts'],
         '#BBDEFB')

# Rendering
draw_box(ax, 9.9, 2.5, 2.8, 3.0, '④ Rendering Stack',
         ['• PBR shaders', '• Glass/transparency', '• Liquid color change', '• Dynamic LCD UI', '• Multi-camera obs.'],
         '#E1BEE7')

# Benchmark
draw_box(ax, 3.5, 0.2, 6.0, 2.0, '⑤ Benchmark (3 Difficulty Tiers)',
         ['Easy: single-step primitives  |  Medium: multi-step protocols  |  Hard: long-horizon experiments'],
         '#FFE0B2')

# Arrows
draw_arrow(ax, 3.1, 3.5, 4.0)
draw_arrow(ax, 6.3, 6.7, 4.0)
draw_arrow(ax, 9.5, 9.9, 4.0)

# Downward arrow to benchmark
ax.annotate('', xy=(6.5, 2.2), xytext=(6.5, 2.5),
            arrowprops=dict(arrowstyle='->', color='#555', lw=2))

plt.tight_layout()
plt.savefig('section2_pipeline.png', bbox_inches='tight')
plt.show()

---
## Section 3 — Custom Physics Plugins

Standard MuJoCo supports rigid-body contact well, but lab instruments rely on **specialised mechanical mechanisms** that require custom simulation. AutoBio implements four physics plugins:

### 3.1 Thread Mechanism (Screw/Unscrew)
Caps on tubes and bottles follow a **helix** trajectory. A point on the cap traces:

$$H(t; r_1, p) = [x, y, z]^\top = [r_1 \cos t,\; r_1 \sin t,\; pt]^\top$$

where $r_1$ is the thread radius and $p$ is the pitch (rise per radian). The robot must simultaneously rotate and translate the cap along this helix.

### 3.2 Detent Mechanism (Click-Stops)
Rotary knobs on centrifuges have discrete **detent positions** — the knob snaps to fixed angular positions. This is modelled with a periodic potential:

$$V(\theta) = -A \cos(n\theta)$$

where $n$ is the number of detent positions and $A$ is the snap strength.

### 3.3 Eccentric Mechanism (Orbital Mixing)
Vortex mixers rotate a plate in an **eccentric** (off-centre) circle, creating shear forces that mix liquid. The centre of the plate traces a circle of radius $r_e$ around the motor axis.

### 3.4 Quasi-Static Liquid
Full fluid simulation is too slow for RL training. AutoBio uses a **quasi-static** liquid model: the liquid surface is assumed flat and the fill level is tracked as a scalar. Colour changes (e.g., after a reagent is added) are handled by blending material properties.

In [ ]:
# ─── Figure 3: Physics Plugins Visualisation ─────────────────────────────────

fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.35)
fig.suptitle('Figure 3 — AutoBio Custom Physics Plugins', fontsize=14, fontweight='bold')

# ──────────── Panel A: Helix (Thread mechanism) ────────────
ax1 = fig.add_subplot(gs[0], projection='3d')
t_vals = np.linspace(0, 4*np.pi, 300)
r1, p  = 0.5, 0.15
x_h = r1 * np.cos(t_vals)
y_h = r1 * np.sin(t_vals)
z_h = p  * t_vals
ax1.plot(x_h, y_h, z_h, 'b-', linewidth=2.5, label='Helix path')

# Mark start, midpoint, end
for idx, color, label in [(0, 'green', 'Start'), (150, 'orange', 'Mid'), (-1, 'red', 'End')]:
    ax1.scatter([x_h[idx]], [y_h[idx]], [z_h[idx]], s=60, c=color, zorder=5)
    ax1.text(x_h[idx]+0.05, y_h[idx]+0.05, z_h[idx]+0.05, label, fontsize=7)

ax1.set_title('A) Thread Mechanism\n(Screw Cap)', fontsize=10)
ax1.set_xlabel('x', fontsize=8); ax1.set_ylabel('y', fontsize=8); ax1.set_zlabel('z (rise)', fontsize=8)
ax1.tick_params(labelsize=7)
ax1.text2D(0.02, 0.02, f'r₁={r1}, p={p}', transform=ax1.transAxes, fontsize=8, color='navy')

# ──────────── Panel B: Detent Potential ────────────
ax2 = fig.add_subplot(gs[1])
theta = np.linspace(-np.pi, np.pi, 500)
n_detents = 6
A = 1.0
V = -A * np.cos(n_detents * theta)
ax2.plot(np.degrees(theta), V, 'purple', linewidth=2)
# Mark stable minima
minima_angles = np.linspace(-180, 180, n_detents, endpoint=False)
for ang in minima_angles:
    V_min = -A * np.cos(n_detents * np.radians(ang))
    ax2.plot(ang, V_min, 'ko', markersize=7)
ax2.set_title('B) Detent Mechanism\n(Click-Stop Knob)', fontsize=10)
ax2.set_xlabel('Knob Angle θ (°)', fontsize=9)
ax2.set_ylabel('Potential V(θ)', fontsize=9)
ax2.set_xticks(np.arange(-180, 181, 60))
ax2.legend(['V(θ) = −A·cos(nθ)', 'Stable detents'], fontsize=8)
ax2.grid(alpha=0.3)
ax2.text(0.02, 0.92, f'n = {n_detents} positions', transform=ax2.transAxes, fontsize=9, color='purple')

# ──────────── Panel C: Eccentric orbit ────────────
ax3 = fig.add_subplot(gs[2])
phi   = np.linspace(0, 2*np.pi, 300)
r_e   = 0.3
r_plate = 0.6

# Motor axis at origin; plate centre traces eccentric circle
plate_cx = r_e * np.cos(phi)
plate_cy = r_e * np.sin(phi)
ax3.plot(plate_cx, plate_cy, 'r--', linewidth=1.5, label='Plate centre orbit')

# Draw plate at a few positions
for angle in [0, np.pi/2, np.pi, 3*np.pi/2]:
    cx = r_e * np.cos(angle)
    cy = r_e * np.sin(angle)
    circle = Circle((cx, cy), r_plate, fill=False, edgecolor='steelblue', linewidth=1.5, alpha=0.5)
    ax3.add_patch(circle)

ax3.plot(0, 0, 'k+', markersize=12, markeredgewidth=2, label='Motor axis')
ax3.set_aspect('equal'); ax3.set_xlim(-1.1, 1.1); ax3.set_ylim(-1.1, 1.1)
ax3.set_title('C) Eccentric Mechanism\n(Vortex Mixer)', fontsize=10)
ax3.set_xlabel('x (m)', fontsize=9); ax3.set_ylabel('y (m)', fontsize=9)
ax3.legend(fontsize=8, loc='lower right'); ax3.grid(alpha=0.3)

# ──────────── Panel D: Quasi-static liquid fill ────────────
ax4 = fig.add_subplot(gs[3])
tube_width  = 0.4
tube_height = 3.0
fill_levels = [0.1, 0.4, 0.7, 1.0]  # Fraction of tube filled
colors_liq  = ['#80D8FF', '#29B6F6', '#0288D1', '#4CAF50']  # Last = after reagent
x_pos = [0.2, 0.8, 1.4, 2.0]
labels_l = ['10%\n(empty)', '40%\n(buffer)', '70%\n(sample)', '100%\n(+reagent)']

for xp, fl, cl, lab in zip(x_pos, fill_levels, colors_liq, labels_l):
    # Tube outline
    tube = FancyBboxPatch((xp, 0), tube_width, tube_height,
                          boxstyle='round,pad=0.02', facecolor='none',
                          edgecolor='black', linewidth=2)
    ax4.add_patch(tube)
    # Liquid fill
    liquid = FancyBboxPatch((xp+0.02, 0.02), tube_width-0.04, (tube_height-0.04)*fl,
                            boxstyle='round,pad=0.01', facecolor=cl, alpha=0.75,
                            edgecolor='none')
    ax4.add_patch(liquid)
    # Fill level label
    ax4.text(xp + tube_width/2, (tube_height-0.04)*fl + 0.15, f'{int(fl*100)}%',
             ha='center', fontsize=8, color='navy')
    ax4.text(xp + tube_width/2, -0.35, lab, ha='center', fontsize=7.5)

# Colour change annotation
ax4.annotate('Colour changes\nafter reagent!',
             xy=(2.2, tube_height*0.8), xytext=(2.55, tube_height*0.6),
             fontsize=8, color='darkgreen',
             arrowprops=dict(arrowstyle='->', color='darkgreen'))

ax4.set_xlim(0, 2.8); ax4.set_ylim(-0.6, tube_height + 0.5)
ax4.set_title('D) Quasi-Static Liquid\n(Fill Level + Colour)', fontsize=10)
ax4.axis('off')

plt.savefig('section3_physics.png', bbox_inches='tight')
plt.show()
print("Tip: Each plugin maps to a real lab instrument — screw caps, centrifuge knobs, vortex mixers, and tubes.")

---
## Section 4 — The AutoBio Benchmark

AutoBio's benchmark decomposes complex experiments into **biological primitives** and groups tasks into three difficulty tiers:

### Biological Primitives
Every task is built from a fixed vocabulary of fundamental lab actions:

| Primitive | Description | Key Challenges |
|---|---|---|
| `open_cap` | Remove screw or flip cap | Thread mechanism, force control |
| `pick_tube` | Pick up an Eppendorf tube | Precision grasp of small objects |
| `pipette` | Aspirate/dispense liquid | High-precision tip alignment |
| `insert_slot` | Insert tube into centrifuge rotor | Sub-mm slot alignment |
| `turn_knob` | Rotate centrifuge speed/time knob | Detent mechanism, counting clicks |
| `press_button` | Press UI button on instrument | Precise fingertip positioning |
| `read_display` | Read LCD display value | Visual reasoning on digital UI |
| `mix_vortex` | Activate vortex mixer | Eccentric orbit + timing |

### Difficulty Tiers

| Tier | # Sub-tasks | Example Task | Key Skill Tested |
|---|---|---|---|
| **Easy** | 1–3 | Open cap, Pick tube | Single primitive, no sequencing |
| **Medium** | 4–8 | Load centrifuge (open → pick → insert → close) | Short horizon + precision |
| **Hard** | 9+ | Run full PCR protocol | Long-horizon + UI interaction + visual reading |

In [ ]:
# ─── Figure 4: Benchmark task taxonomy and workflow example ───────────────────

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Figure 4 — AutoBio Benchmark Overview', fontsize=14, fontweight='bold')

# ──── Panel A: Task difficulty distribution ────
ax = axes[0]
primitives = ['open_cap', 'pick_tube', 'pipette', 'insert_slot',
              'turn_knob', 'press_button', 'read_display', 'mix_vortex']
difficulty_presence = {
    'Easy':   [1, 1, 0, 0, 0, 0, 0, 0],
    'Medium': [1, 1, 1, 1, 1, 0, 0, 0],
    'Hard':   [1, 1, 1, 1, 1, 1, 1, 1],
}
tier_colors = {'Easy': '#A5D6A7', 'Medium': '#FFE082', 'Hard': '#EF9A9A'}

x = np.arange(len(primitives))
width = 0.25
for i, (tier, vals) in enumerate(difficulty_presence.items()):
    bars = ax.bar(x + i*width, vals, width, label=tier, color=tier_colors[tier],
                  edgecolor='black', linewidth=0.7)

ax.set_xticks(x + width)
ax.set_xticklabels(primitives, rotation=35, ha='right', fontsize=9)
ax.set_yticks([0, 1])
ax.set_yticklabels(['Not present', 'Present'], fontsize=9)
ax.set_title('A) Which Primitives Appear\nin Each Difficulty Tier', fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(-0.1, 1.5)
ax.grid(axis='y', alpha=0.3)

# ──── Panel B: Workflow graph for a Medium task ("Load Centrifuge") ────
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 7)
ax2.axis('off')
ax2.set_title('B) Example Medium Task: "Load Centrifuge"', fontsize=11)

steps = [
    (1.0, 6.0, 'open_cap\n(tube)', '#C8E6C9'),
    (3.5, 6.0, 'pick_tube', '#C8E6C9'),
    (6.0, 6.0, 'insert_slot\n(centrifuge)', '#FFE0B2'),
    (8.5, 6.0, 'close_lid\n(centrifuge)', '#FFE0B2'),
    (3.5, 3.5, 'turn_knob\n(speed)', '#BBDEFB'),
    (6.0, 3.5, 'turn_knob\n(time)', '#BBDEFB'),
    (8.5, 3.5, 'press_button\n(start)', '#E1BEE7'),
    (5.0, 1.2, '✅ DONE\n(centrifuge running)', '#F0F4C3'),
]

for (xc, yc, label, col) in steps:
    box = FancyBboxPatch((xc-1.0, yc-0.55), 2.0, 1.1, boxstyle='round,pad=0.1',
                         facecolor=col, edgecolor='#444', linewidth=1.2)
    ax2.add_patch(box)
    ax2.text(xc, yc, label, ha='center', va='center', fontsize=8.5)

# Arrows
edges = [(0,1),(1,2),(2,3),(1,4),(4,5),(5,6),(3,6),(6,7),(5,7)]
for (i,j) in edges:
    x1, y1, *_ = steps[i]; x2, y2, *_ = steps[j]
    ax2.annotate('', xy=(x2-1.05, y2), xytext=(x1+1.05, y1),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=1.3,
                                 connectionstyle='arc3,rad=0.0'))

# Tier labels
ax2.text(0.1, 6.0, 'Step 1-2', fontsize=8, color='#2E7D32', style='italic')
ax2.text(0.1, 3.5, 'Step 3-5', fontsize=8, color='#1565C0', style='italic')
ax2.text(0.1, 1.2, 'Goal',     fontsize=8, color='#F57F17', style='italic')

plt.tight_layout()
plt.savefig('section4_benchmark.png', bbox_inches='tight')
plt.show()

---
## Section 5 — Evaluation Metrics

AutoBio uses a hierarchical success metric that separately rewards sub-task completion, avoiding the common all-or-nothing evaluation pitfall.

### 5.1 Success Rate (SR)

The binary task-level success — did the robot complete **all** sub-tasks correctly within the time limit?

$$\text{SR} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{1}[\text{task}_i \text{ fully completed}]$$

### 5.2 Sub-Task Success Rate (SSR)

Awards partial credit by measuring what fraction of sub-tasks were completed:

$$\text{SSR} = \frac{1}{N} \sum_{i=1}^{N} \frac{\text{sub-tasks completed}_i}{\text{total sub-tasks}_i}$$

### 5.3 Why Both Matter

- A model that always fails on the **last** sub-task scores **0 SR** but high **SSR** — this reveals a specific failure mode.
- In long-horizon tasks, SSR gives a much richer signal than SR alone.

### 5.4 Observation Space

Each observation step returns:
- **RGB image** from wrist camera (256×256)
- **RGB image** from top-down camera (256×256)  
- **Proprioception** — 7-DoF joint angles + gripper state
- **Language instruction** — free-form text describing the task

In [ ]:
# ─── Figure 5: Metrics illustration ──────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 5 — Understanding SR vs SSR Metrics', fontsize=13, fontweight='bold')

# ──── Panel A: SR vs SSR comparison ────
ax = axes[0]
scenarios = ['Model A\n(fails early)', 'Model B\n(fails at end)', 'Model C\n(perfect)']
sr_vals   = [0.0,   0.0,   1.0]
ssr_vals  = [0.25,  0.85,  1.0]

x = np.arange(len(scenarios))
w = 0.3
bars1 = ax.bar(x - w/2, sr_vals,  w, label='SR  (Task-Level)',    color='#F44336', edgecolor='black')
bars2 = ax.bar(x + w/2, ssr_vals, w, label='SSR (Sub-Task-Level)', color='#4CAF50', edgecolor='black')

for bar, val in zip(list(bars1)+list(bars2), sr_vals+ssr_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}',
            ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=10)
ax.set_ylim(0, 1.25)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('A) SR vs SSR:\nWhy Both Metrics Matter', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Annotation
ax.annotate('SR alone would rate\nA and B equally bad!',
            xy=(0.5, 0.05), xytext=(0.5, 0.5),
            ha='center', fontsize=9, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred'))

# ──── Panel B: Observation modalities ────
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 7); ax2.axis('off')
ax2.set_title('B) VLA Model Observation Space', fontsize=11)

# Draw observation boxes
obs_boxes = [
    (1.5, 5.2, 2.2, 1.2, 'Wrist Camera\n(256×256 RGB)', '#BBDEFB'),
    (4.5, 5.2, 2.2, 1.2, 'Top-down Camera\n(256×256 RGB)', '#BBDEFB'),
    (7.5, 5.2, 2.0, 1.2, 'Proprioception\n(7-DoF joints)', '#C8E6C9'),
    (4.5, 3.2, 4.5, 1.2, 'Language Instruction\n"Insert tube into centrifuge slot"', '#FFE0B2'),
]
for (xc, yc, w, h, lab, col) in obs_boxes:
    box = FancyBboxPatch((xc-w/2, yc-h/2), w, h, boxstyle='round,pad=0.1',
                         facecolor=col, edgecolor='#555', linewidth=1.3)
    ax2.add_patch(box)
    ax2.text(xc, yc, lab, ha='center', va='center', fontsize=8.5)

# Arrow to VLA
vla_box = FancyBboxPatch((3.2, 0.8), 3.6, 1.3, boxstyle='round,pad=0.1',
                         facecolor='#E1BEE7', edgecolor='#333', linewidth=2)
ax2.add_patch(vla_box)
ax2.text(5.0, 1.45, 'VLA Model\n(OpenVLA / RDT-1B)', ha='center', va='center', fontsize=10, fontweight='bold')

for src_xy in [(1.5, 4.6), (4.5, 4.6), (7.5, 4.6), (4.5, 2.6)]:
    ax2.annotate('', xy=(5.0, 2.1), xytext=src_xy,
                 arrowprops=dict(arrowstyle='->', color='#555', lw=1.2,
                                 connectionstyle='arc3,rad=0.1'))

# Output
ax2.annotate('7-DoF action\n(Δpose + gripper)',
             xy=(5.0, 0.5), xytext=(5.0, 0.8),
             ha='center', fontsize=9, color='#4527A0',
             arrowprops=dict(arrowstyle='->', color='#4527A0'))

plt.tight_layout()
plt.savefig('section5_metrics.png', bbox_inches='tight')
plt.show()

---
## Section 6 — Baseline Results and Analysis

AutoBio evaluates two state-of-the-art VLA models:

| Model | Architecture | Training Data | Notes |
|---|---|---|---|
| **OpenVLA** | LLaMA-7B backbone + visual encoder | Open-X embodiment | Open-source; general-purpose |
| **RDT-1B** | Diffusion transformer (1B params) | RT-1 + in-house data | Strong bimanual; flow-matching |

Both models are fine-tuned on AutoBio demonstrations generated by a **scripted oracle policy** using the built-in teleoperation interface.

### Key Findings from the Paper

1. **Both models fail almost entirely on Hard tasks** — SSR < 15% for long-horizon protocols.
2. **Precision manipulation is the bottleneck** — slot insertion and pipette alignment are particularly poor.
3. **Visual reasoning over displays is unsolved** — models rarely correctly read LCD values.
4. **Easy tasks are learnable** — both models reach ~40–60% SR on single-primitive tasks with sufficient demos.

These results confirm that AutoBio successfully exposes critical gaps not covered by existing benchmarks.

In [ ]:
# ─── Figure 6: Simulated Baseline Results ────────────────────────────────────
# NOTE: These values are illustrative, consistent with the paper's qualitative findings.
# Refer to the paper (Table 1) for exact numbers.

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figure 6 — Baseline VLA Model Performance on AutoBio\n(Illustrative — see paper Table 1 for exact values)',
             fontsize=13, fontweight='bold')

tiers   = ['Easy', 'Medium', 'Hard']
models  = ['OpenVLA', 'RDT-1B']
colors  = {'OpenVLA': '#42A5F5', 'RDT-1B': '#EF5350'}

# Illustrative SR and SSR values (qualitatively aligned with paper findings)
sr_data  = {'OpenVLA': [45, 15, 5],  'RDT-1B': [58, 22, 8]}
ssr_data = {'OpenVLA': [62, 38, 14], 'RDT-1B': [71, 48, 19]}

x = np.arange(len(tiers))
width = 0.35

# SR
ax = axes[0]
for i, model in enumerate(models):
    bars = ax.bar(x + i*width, sr_data[model], width, label=model,
                  color=colors[model], edgecolor='black', linewidth=0.7, alpha=0.85)
    for bar, val in zip(bars, sr_data[model]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1.5, f'{val}%',
                ha='center', fontsize=9)
ax.set_xticks(x + width/2)
ax.set_xticklabels(tiers, fontsize=11)
ax.set_ylabel('Success Rate (%)', fontsize=10)
ax.set_title('A) Task-Level SR', fontsize=11)
ax.set_ylim(0, 85); ax.legend(); ax.grid(axis='y', alpha=0.3)

# SSR
ax = axes[1]
for i, model in enumerate(models):
    bars = ax.bar(x + i*width, ssr_data[model], width, label=model,
                  color=colors[model], edgecolor='black', linewidth=0.7, alpha=0.85)
    for bar, val in zip(bars, ssr_data[model]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1.5, f'{val}%',
                ha='center', fontsize=9)
ax.set_xticks(x + width/2)
ax.set_xticklabels(tiers, fontsize=11)
ax.set_ylabel('Sub-Task SR (%)', fontsize=10)
ax.set_title('B) Sub-Task Level SSR', fontsize=11)
ax.set_ylim(0, 95); ax.legend(); ax.grid(axis='y', alpha=0.3)

# Failure mode breakdown
ax = axes[2]
failure_modes = ['Precision\nManipulation', 'Visual\nReasoning\n(LCD)', 'Instruction\nFollowing', 'Long-Horizon\nPlanning']
failure_pct   = [48, 24, 16, 12]
explode = (0.08, 0.05, 0, 0)
fail_colors = ['#EF5350', '#FF9800', '#FFC107', '#8BC34A']
wedges, texts, autotexts = ax.pie(failure_pct, labels=failure_modes, autopct='%1.0f%%',
                                  colors=fail_colors, explode=explode,
                                  startangle=140, textprops={'fontsize': 9})
for at in autotexts:
    at.set_fontsize(9); at.set_fontweight('bold')
ax.set_title('C) Root Cause of Failures\n(Hard Tier)', fontsize=11)

plt.tight_layout()
plt.savefig('section6_results.png', bbox_inches='tight')
plt.show()
print("Key insight: Precision manipulation accounts for ~48% of all failures — the dominant bottleneck.")

---
## Section 7 — Deep Dive: The Thread Mechanism in Detail

One of AutoBio's most distinctive contributions is simulating **screw-cap mechanisms** on lab tubes. This section lets you explore the helix geometry and understand how the simulation generates the correct force–torque coupling.

### Mathematical Background

A bounded helix with radius $r_1$ and pitch $p$ over $n$ full turns is parameterised as:

$$H(t) = \begin{bmatrix} r_1 \cos t \\ r_1 \sin t \\ pt \end{bmatrix}, \quad t \in [0, 2\pi n]$$

The **arc length** along the helix up to parameter $t$ is:

$$s(t) = t \sqrt{r_1^2 + p^2}$$

The **lead angle** (helix angle with respect to the horizontal) is:

$$\psi = \arctan\left(\frac{p}{r_1}\right)$$

A robot screwing off a cap must apply both a **torque** (rotation) and an **upward force** (translation) in the ratio dictated by the lead angle.

In [ ]:
# ─── Interactive: Explore helix parameters ────────────────────────────────────

def plot_helix_analysis(r1=0.5, p=0.12, n_turns=2):
    """Plot helix geometry and key derived quantities."""
    t = np.linspace(0, 2*np.pi*n_turns, 600)
    x = r1 * np.cos(t)
    y = r1 * np.sin(t)
    z = p  * t
    s = t  * np.sqrt(r1**2 + p**2)   # arc length
    psi = np.degrees(np.arctan(p / r1))

    fig = plt.figure(figsize=(14, 5))
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
    fig.suptitle(f'Thread Mechanism Analysis  |  r₁={r1} m, pitch p={p} m/rad, {n_turns} turns',
                 fontsize=12, fontweight='bold')

    # 3-D helix
    ax1 = fig.add_subplot(gs[0], projection='3d')
    sc = ax1.scatter(x, y, z, c=t, cmap='plasma', s=8)
    ax1.plot(x, y, z, 'k-', lw=0.5, alpha=0.3)
    plt.colorbar(sc, ax=ax1, label='t (rad)', shrink=0.7, pad=0.1)
    ax1.set_title('Helix Trajectory', fontsize=10)
    ax1.set_xlabel('x (m)'); ax1.set_ylabel('y (m)'); ax1.set_zlabel('z (m)')
    ax1.text2D(0.02, 0.92, f'Lead angle ψ = {psi:.1f}°', transform=ax1.transAxes, fontsize=9)
    ax1.text2D(0.02, 0.85, f'Total rise = {p*2*np.pi*n_turns:.3f} m', transform=ax1.transAxes, fontsize=9)

    # Arc length vs t
    ax2 = fig.add_subplot(gs[1])
    ax2.plot(t, s, 'royalblue', linewidth=2.5)
    ax2.fill_between(t, s, alpha=0.15, color='royalblue')
    ax2.set_xlabel('Parameter t (rad)', fontsize=10)
    ax2.set_ylabel('Arc Length s(t) (m)', fontsize=10)
    ax2.set_title(f'Arc Length vs t\n(slope = √(r₁²+p²) = {np.sqrt(r1**2+p**2):.3f})', fontsize=10)
    ax2.grid(alpha=0.3)
    # Show linearity
    ax2.plot(t, t*np.sqrt(r1**2+p**2), 'r--', lw=1, label='Linear approx')
    ax2.legend(fontsize=8)

    # Rise vs rotation
    ax3 = fig.add_subplot(gs[2])
    rotations = t / (2*np.pi)   # in full turns
    rise = p * t
    ax3.plot(rotations, rise*1000, 'darkorange', linewidth=2.5)
    ax3.set_xlabel('Cap Rotation (full turns)', fontsize=10)
    ax3.set_ylabel('Cap Rise (mm)', fontsize=10)
    ax3.set_title(f'Rise per Turn = {p*2*np.pi*1000:.1f} mm/turn', fontsize=10)
    ax3.grid(alpha=0.3)

    # Add annotation for one full turn
    ax3.annotate(f'+{p*2*np.pi*1000:.1f} mm',
                 xy=(1.0, p*2*np.pi*1000), xytext=(1.2, p*2*np.pi*1000*0.6),
                 fontsize=9, color='darkorange',
                 arrowprops=dict(arrowstyle='->', color='darkorange'))

    plt.savefig('section7_thread.png', bbox_inches='tight')
    plt.show()
    print(f"Summary: Lead angle = {psi:.2f}°, Total arc length = {s[-1]:.4f} m, "
          f"Total rise = {rise[-1]*1000:.2f} mm")

# ── Run with default parameters first ──
plot_helix_analysis(r1=0.5, p=0.12, n_turns=2)

In [ ]:
# ── Try different parameters to see the effect ──
# Fine thread (small pitch) — harder to unscrew, more turns needed
plot_helix_analysis(r1=0.4, p=0.05, n_turns=4)

# Coarse thread (large pitch) — fewer turns, more rise per turn
# plot_helix_analysis(r1=0.5, p=0.25, n_turns=1)

---
## Section 8 — Long-Horizon Task Concatenation

AutoBio's Hard tasks are built by **concatenating** sub-policies. The paper shows that even when individual sub-tasks succeed at moderate rates, the **joint success probability** of a long-horizon task drops sharply.

If each sub-task has independent success rate $p$, and a task has $n$ sub-tasks:

$$\text{SR}_{\text{task}} = p^n$$

This is the **compounding error problem**. AutoBio evaluates whether VLA models can maintain accuracy across many steps — a much harder bar than single-step benchmarks.

In [ ]:
# ─── Figure 8: Compounding error in long-horizon tasks ───────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 8 — The Compounding Error Problem in Long-Horizon Tasks',
             fontsize=13, fontweight='bold')

n_steps = np.arange(1, 21)
sub_task_probs = [0.95, 0.85, 0.75, 0.60]
colors_p = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
labels_p = [f'Per-step SR = {p:.0%}' for p in sub_task_probs]

# ──── Panel A: Compounding effect ────
ax = axes[0]
for p, col, lab in zip(sub_task_probs, colors_p, labels_p):
    joint_sr = p ** n_steps
    ax.plot(n_steps, joint_sr * 100, 'o-', color=col, linewidth=2, label=lab, markersize=4)

# Mark AutoBio difficulty tiers
for tier, n, col in [('Easy (≤3)', 3, '#888'), ('Medium (≤8)', 8, '#555'), ('Hard (≤18)', 18, '#222')]:
    ax.axvline(x=n, color=col, linestyle=':', linewidth=1.5)
    ax.text(n+0.2, 80, tier, color=col, fontsize=8, rotation=90)

ax.set_xlabel('Number of Sub-Tasks (n)', fontsize=11)
ax.set_ylabel('Joint Task SR (%)', fontsize=11)
ax.set_title('A) Compounding Error: SR = p^n', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(1, 20); ax.set_ylim(0, 105)

# ──── Panel B: AutoBio model performance vs horizon ────
ax2 = axes[1]
# Illustrative data — models improve with shorter horizon (fewer steps)
horizon_bins = ['1–3 steps\n(Easy)', '4–8 steps\n(Medium)', '9–18 steps\n(Hard)']
openvla_ssr  = [62, 38, 14]
rdt1b_ssr    = [71, 48, 19]

x = np.arange(len(horizon_bins))
w = 0.3
ax2.bar(x - w/2, openvla_ssr, w, label='OpenVLA', color='#42A5F5', edgecolor='black', linewidth=0.7)
ax2.bar(x + w/2, rdt1b_ssr,  w, label='RDT-1B',  color='#EF5350', edgecolor='black', linewidth=0.7)

# Fit trend lines
for data, col, lab in [(openvla_ssr, '#0D47A1', 'OpenVLA trend'),
                       (rdt1b_ssr,   '#B71C1C', 'RDT-1B trend')]:
    z = np.polyfit(x, data, 1)
    p_fn = np.poly1d(z)
    ax2.plot(x, p_fn(x), '--', color=col, linewidth=1.5, label=lab)

ax2.set_xticks(x)
ax2.set_xticklabels(horizon_bins, fontsize=9)
ax2.set_ylabel('Sub-Task SR (%)', fontsize=11)
ax2.set_title('B) Observed Model SSR\nvs Task Horizon', fontsize=11)
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0, 90)

plt.tight_layout()
plt.savefig('section8_longhorizon.png', bbox_inches='tight')
plt.show()
print("Finding: Even with 85% per-step accuracy, a 10-step task has only ~20% joint success.")

---
## Section 9 — Rendering: Transparent Materials Challenge

Biology labs are full of **transparent** objects — glass beakers, clear tubes, liquid samples. Standard rendering engines use opaque shaders, making transparent objects appear as solid blobs or invisible. AutoBio uses **Physically-Based Rendering (PBR)** with custom shaders for:

- **Refraction** — light bends as it passes through glass
- **Fresnel effect** — reflection increases at grazing angles
- **Liquid meniscus** — curved surface at air-liquid boundary
- **Caustics** — light focusing patterns on surfaces below glass

Below we simulate the **Fresnel effect** — a core visual cue that makes glass identifiable — and show why it matters for a vision model trying to estimate liquid level.

In [ ]:
# ─── Figure 9: Fresnel effect & transparency rendering ───────────────────────

def fresnel_reflectance(n1, n2, theta_i_deg):
    """Compute Fresnel reflectance for unpolarised light.
    n1, n2: refractive indices; theta_i: angle of incidence (degrees).
    """
    theta_i = np.radians(theta_i_deg)
    sin_theta_t = (n1 / n2) * np.sin(theta_i)
    # Total internal reflection
    tir_mask = sin_theta_t > 1.0
    sin_theta_t = np.clip(sin_theta_t, -1.0, 1.0)
    theta_t = np.arcsin(sin_theta_t)

    cos_i = np.cos(theta_i)
    cos_t = np.cos(theta_t)

    Rs = ((n1*cos_i - n2*cos_t) / (n1*cos_i + n2*cos_t))**2
    Rp = ((n2*cos_i - n1*cos_t) / (n2*cos_i + n1*cos_t))**2
    R  = 0.5 * (Rs + Rp)
    R[tir_mask] = 1.0
    return R

angles = np.linspace(0, 90, 500)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figure 9 — Rendering Challenges: Transparent Lab Objects', fontsize=13, fontweight='bold')

# ──── Panel A: Fresnel curves for glass and water ────
ax = axes[0]
materials = [
    ('Glass (n=1.5)', 1.0, 1.5, '#4FC3F7'),
    ('Water (n=1.33)', 1.0, 1.33, '#26C6DA'),
    ('Sapphire (n=1.76)', 1.0, 1.76, '#7E57C2'),
]
for label, n1, n2, col in materials:
    R = fresnel_reflectance(n1, n2, angles)
    ax.plot(angles, R, color=col, linewidth=2.5, label=label)

ax.set_xlabel('Angle of Incidence (°)', fontsize=10)
ax.set_ylabel('Reflectance', fontsize=10)
ax.set_title('A) Fresnel Reflectance\n(makes glass recognisable!)', fontsize=10)
ax.legend(fontsize=9)
ax.axvline(x=90, color='red', linestyle='--', lw=1, label='Grazing angle')
ax.text(75, 0.3, 'High\nreflection at\ngrazing angles', fontsize=8, color='darkred')
ax.grid(alpha=0.3)
ax.set_xlim(0, 90); ax.set_ylim(0, 1.05)

# ──── Panel B: Refraction geometry ────
ax2 = axes[1]
ax2.set_xlim(-3, 3); ax2.set_ylim(-3, 3); ax2.axis('off')
ax2.set_title('B) Refraction Through Glass Tube\n(Light Bends at Interface)', fontsize=10)

# Glass tube boundary
ax2.axhline(y=0, color='steelblue', linewidth=3, alpha=0.5, label='Air-glass interface')
ax2.axhline(y=-0.5, color='steelblue', linewidth=3, alpha=0.5)
ax2.fill_between([-3, 3], 0, -0.5, color='lightblue', alpha=0.3)

# Incident ray
theta_inc = 45
ax2.annotate('', xy=(0, 0), xytext=(-1.5, 1.5),
             arrowprops=dict(arrowstyle='->', color='orange', lw=2))
ax2.text(-1.8, 1.6, 'Incident ray', color='orange', fontsize=8)

# Refracted ray (snell's law: n1*sin(θ1) = n2*sin(θ2))
n_air, n_glass = 1.0, 1.5
theta_r = np.degrees(np.arcsin(n_air / n_glass * np.sin(np.radians(theta_inc))))
dx_refr = 0.5 * np.tan(np.radians(theta_r))
ax2.annotate('', xy=(dx_refr, -0.5), xytext=(0, 0),
             arrowprops=dict(arrowstyle='->', color='royalblue', lw=2))
ax2.text(dx_refr+0.1, -0.65, f'Refracted (θ={theta_r:.1f}°)', color='royalblue', fontsize=8)

# Exiting ray (parallel to incident)
ax2.annotate('', xy=(dx_refr+1.5, -2.0), xytext=(dx_refr, -0.5),
             arrowprops=dict(arrowstyle='->', color='orange', lw=2, linestyle='dashed'))
ax2.text(dx_refr+0.6, -1.5, 'Exiting ray\n(parallel shift)', color='orange', fontsize=8)

ax2.text(0.1, 1.0, 'Air (n=1.0)', fontsize=9, color='steelblue')
ax2.text(0.1, -0.25, 'Glass (n=1.5)', fontsize=9, color='steelblue')
ax2.text(0.1, -1.2, 'Air (n=1.0)', fontsize=9, color='steelblue')

# Angle annotations
ax2.text(-0.7, 0.3, f'θᵢ={theta_inc}°', fontsize=8)
ax2.text(0.05, -0.15, f'θᵣ={theta_r:.1f}°', fontsize=8)

# ──── Panel C: Why this matters for liquid-level estimation ────
ax3 = axes[2]
ax3.set_xlim(0, 6); ax3.set_ylim(0, 6); ax3.axis('off')
ax3.set_title('C) Visual Ambiguity in\nTransparent Tube Estimation', fontsize=10)

# Two tubes side by side
for x_pos, fill_pct, renderer, col_liq, col_text in [
    (0.5, 0.6, 'Standard\n(opaque)', '#90CAF9', '#1565C0'),
    (3.0, 0.6, 'AutoBio PBR\n(transparent)', '#90CAF9', '#1565C0')
]:
    # Tube
    tube = FancyBboxPatch((x_pos, 0.5), 1.8, 4.5, boxstyle='round,pad=0.05',
                          facecolor='white' if 'PBR' in renderer else '#EEEEEE',
                          edgecolor='black', linewidth=2, alpha=0.9)
    ax3.add_patch(tube)

    # Liquid fill
    liq = FancyBboxPatch((x_pos+0.1, 0.6), 1.6, 4.3*fill_pct,
                         boxstyle='round,pad=0.02',
                         facecolor=col_liq,
                         alpha=0.3 if 'PBR' in renderer else 0.9,
                         edgecolor='none')
    ax3.add_patch(liq)

    if 'PBR' in renderer:
        # Add meniscus line
        ax3.plot([x_pos+0.1, x_pos+1.7],
                 [0.6 + 4.3*fill_pct, 0.6 + 4.3*fill_pct],
                 color='steelblue', linewidth=2.5, label='Meniscus')
        ax3.text(x_pos+0.9, 0.6+4.3*fill_pct+0.15, 'Meniscus visible!',
                 ha='center', fontsize=8, color='steelblue')

    ax3.text(x_pos+0.9, 5.3, renderer, ha='center', fontsize=9, fontweight='bold')
    ax3.text(x_pos+0.9, 0.1, f'{int(fill_pct*100)}% full', ha='center', fontsize=9)

ax3.text(2.5, 2.8, '→', fontsize=24, ha='center', color='gray')

plt.tight_layout()
plt.savefig('section9_rendering.png', bbox_inches='tight')
plt.show()
print("PBR rendering makes the liquid surface (meniscus) visible, critical for pipetting accuracy.")

---
## Section 10 — Comparison with Prior Benchmarks

AutoBio situates itself in a landscape of existing robot learning benchmarks. The key differentiators are:

| Feature | Meta-World | LIBERO | RoboTwin | Chemistry3D | **AutoBio** |
|---|---|---|---|---|---|
| Domain | Factory | Home | Home | Chemistry lab | **Biology lab** |
| Task horizon | Short | Short-Med | Medium | Medium | **Long** |
| Precision required | Low | Low | Medium | Medium | **Very high** |
| Transparent objects | ✗ | ✗ | ✗ | ✗ | **✓** |
| Digital UI interaction | ✗ | ✗ | ✗ | ✗ | **✓** |
| Language-guided | ✗ | ✓ | ✓ | ✓ | **✓** |
| VLA model integration | Partial | ✓ | ✓ | ✓ | **✓** |
| Biologically grounded | ✗ | ✗ | ✗ | Partial | **✓** |

In [ ]:
# ─── Figure 10: Feature comparison radar chart ───────────────────────────────

fig, ax = plt.subplots(1, 1, figsize=(8, 8), subplot_kw=dict(polar=True))
fig.suptitle('Figure 10 — Benchmark Feature Comparison (Radar Chart)', fontsize=13, fontweight='bold')

dimensions = [
    'Precision\nRequirement',
    'Task\nHorizon',
    'Transparent\nMaterials',
    'Digital UI\nInteraction',
    'Language\nGrounding',
    'Biological\nRealism',
]
N = len(dimensions)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

benchmarks = {
    'Meta-World':  [2, 2, 1, 1, 2, 1],
    'LIBERO':      [2, 3, 1, 1, 4, 1],
    'Chemistry3D': [3, 3, 2, 2, 3, 3],
    'AutoBio':     [5, 5, 5, 5, 5, 5],
}
bm_colors = {'Meta-World': '#9E9E9E', 'LIBERO': '#42A5F5',
             'Chemistry3D': '#FF9800', 'AutoBio': '#E91E63'}
bm_lw = {'Meta-World': 1.5, 'LIBERO': 1.5, 'Chemistry3D': 1.5, 'AutoBio': 3}

for name, vals in benchmarks.items():
    closed = vals + vals[:1]
    ax.plot(angles, closed, 'o-', color=bm_colors[name], linewidth=bm_lw[name],
            label=name, markersize=5 if name == 'AutoBio' else 3)
    ax.fill(angles, closed, alpha=0.06 if name != 'AutoBio' else 0.15,
            color=bm_colors[name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dimensions, fontsize=10)
ax.set_ylim(0, 5)
ax.set_yticks([1,2,3,4,5])
ax.set_yticklabels(['1','2','3','4','5'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)

plt.tight_layout()
plt.savefig('section10_comparison.png', bbox_inches='tight')
plt.show()

---
## Section 11 — Summary & Key Takeaways

AutoBio makes four concrete contributions to the robotics community:

1. **Simulation Framework** — A MuJoCo-based simulator with custom physics plugins (thread, detent, eccentric, quasi-static liquid) and PBR rendering for transparent materials and dynamic UI.

2. **Asset Digitization Pipeline** — An end-to-end workflow for converting real lab instruments into simulation-ready 3-D assets.

3. **Benchmark** — 3-tier (Easy/Medium/Hard) biologically grounded tasks with SR and SSR metrics, demonstration generation tooling, and out-of-the-box VLA integration.

4. **Baseline Evaluation** — Reveals that state-of-the-art VLA models (OpenVLA, RDT-1B) fail significantly on precision manipulation, LCD visual reasoning, and long-horizon instruction following — motivating future research directions.

### Open Research Questions

- Can **foundation models** (GPT-4V, Gemini) improve LCD-reading sub-tasks when used as perception modules?
- Does **imitation learning with human teleoperation** outperform scripted oracle demonstrations?
- How well do policies transfer from **simulation to real lab hardware** (sim-to-real gap)?
- Can **task graph planning** solve long-horizon compound failures?
- Would **active sensing** (moving the camera to reduce ambiguity) help with transparent-object tasks?

---

### References

- Lan et al. (2025). *AutoBio: A Simulation and Benchmark for Robotic Automation in Digital Biology Laboratory*. arXiv:2505.14030.
- Kim et al. (2024). *OpenVLA: An Open-Source Vision-Language-Action Model*. arXiv:2406.09246.
- Liu et al. (2025). *RDT-1B: A Diffusion Foundation Model for Bimanual Manipulation*. ICLR 2025.
- Todorov et al. (2012). *MuJoCo: A physics engine for model-based control*. IROS 2012.
- Mu et al. (2025). *RoboTwin: Dual-Arm Robot Benchmark with Generative Digital Twins*. arXiv:2504.13059.

---
## Section 12 — Exercises

Test your understanding with these guided exercises. Solutions are provided in the cells below each exercise.

---

### Exercise 1 — Thread Mechanism
A centrifuge tube cap has **3 full turns** to remove, with thread radius $r_1 = 0.6$ cm and pitch $p = 0.08$ cm/rad.

a) What is the total **rise** (axial displacement) when the cap is removed?  
b) What is the **arc length** of the helix traced by the cap?  
c) What is the **lead angle** $\psi$?

In [ ]:
# ── Exercise 1 Solution ──────────────────────────────────────────────────────
r1 = 0.6   # cm
p  = 0.08  # cm/rad
n_turns = 3
t_total = 2 * np.pi * n_turns

rise       = p * t_total
arc_length = t_total * np.sqrt(r1**2 + p**2)
psi        = np.degrees(np.arctan(p / r1))

print(f"(a) Total rise       = {rise:.4f} cm = {rise*10:.4f} mm")
print(f"(b) Arc length       = {arc_length:.4f} cm")
print(f"(c) Lead angle ψ     = {psi:.2f}°")
print()
print("Interpretation: A 7.6° lead angle means for every cm moved along the helix,")
print(f"only {p*10:.2f} mm of axial rise is gained — a fine thread requires many turns.")

---

### Exercise 2 — Compounding Errors
Suppose a VLA model achieves **80% per-step SSR** on individual sub-tasks.

a) What is the joint SR for an Easy task (3 steps)?  
b) What is the joint SR for a Hard task (15 steps)?  
c) What minimum per-step SR is needed to achieve **50% joint SR** on a 10-step task?

In [ ]:
# ── Exercise 2 Solution ──────────────────────────────────────────────────────
p_step = 0.80

sr_easy = p_step ** 3
sr_hard = p_step ** 15
p_needed_10 = 0.50 ** (1/10)

print(f"(a) Easy task SR  (3 steps)  = {sr_easy:.4f} = {sr_easy*100:.2f}%")
print(f"(b) Hard task SR  (15 steps) = {sr_hard:.4f} = {sr_hard*100:.2f}%")
print(f"(c) Min per-step SR for 50%\n    joint SR on 10-step task = {p_needed_10:.4f} = {p_needed_10*100:.2f}%")
print()
print("Insight: A model with 80% per-step accuracy has only 3.5% joint SR on a 15-step task.")
print("To reliably complete 10-step tasks, you need ~93% per-step accuracy!")

# Visualise
steps = np.arange(1, 20)
plt.figure(figsize=(8, 4))
plt.plot(steps, p_step**steps * 100, 'royalblue', linewidth=2.5, label=f'p={p_step}')
plt.plot(steps, p_needed_10**steps * 100, 'darkorange', linewidth=2.5, label=f'p={p_needed_10:.2f}')
plt.axhline(50, color='red', linestyle='--', linewidth=1.5, label='50% target')
plt.axvline(10, color='gray', linestyle=':', linewidth=1.5)
plt.xlabel('Number of Steps'); plt.ylabel('Joint SR (%)')
plt.title('Exercise 2: Compounding Error for Different Per-Step SR')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---

### Exercise 3 — Detent Mechanism
A centrifuge speed knob has **12 detent positions** (one every 30°) with snap strength $A = 2.0$.

a) Plot the potential $V(\theta) = -A\cos(n\theta)$ for $\theta \in [-\pi, \pi]$.  
b) Compute the **torque** $\tau = -dV/d\theta$ and identify where it is maximum.  
c) At what angle does the detent transition occur (unstable equilibrium)?

In [ ]:
# ── Exercise 3 Solution ──────────────────────────────────────────────────────
n_det = 12
A_det = 2.0
theta = np.linspace(-np.pi, np.pi, 1000)

V      = -A_det * np.cos(n_det * theta)
torque = -np.gradient(V, theta)   # τ = -dV/dθ  (restoring torque)

# Max torque
idx_max = np.argmax(np.abs(torque))
theta_max_torque = np.degrees(theta[idx_max])

# Unstable equilibria at peaks of V → cos(nθ) = -1 → nθ = ±π, ±3π, ...
theta_unstable = np.degrees(np.pi / n_det)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Exercise 3 — Detent Mechanism Analysis', fontsize=12, fontweight='bold')

ax = axes[0]
ax.plot(np.degrees(theta), V, 'purple', linewidth=2)
stable   = theta[V == V.min()]
unstable = theta[V == V.max()]
ax.axhline(y=-A_det, color='green',  linestyle='--', label=f'Stable min (V={-A_det})')
ax.axhline(y= A_det, color='red',    linestyle='--', label=f'Unstable max (V={A_det})')
ax.set_xlabel('θ (°)'); ax.set_ylabel('V(θ)')
ax.set_title(f'Potential: V(θ) = −{A_det}·cos({n_det}θ)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(np.degrees(theta), torque, 'darkorange', linewidth=2)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axvline(theta_max_torque, color='red', linestyle=':', label=f'Max |τ| at θ≈{theta_max_torque:.1f}°')
ax2.set_xlabel('θ (°)'); ax2.set_ylabel('Torque τ = −dV/dθ')
ax2.set_title('Restoring Torque'); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

print(f"(a) Plot shown above.")
print(f"(b) Max |torque| occurs at θ ≈ ±{theta_max_torque:.1f}° (between detent positions).")
print(f"(c) Unstable equilibrium at θ = ±{theta_unstable:.2f}° from each stable detent.")
plt.tight_layout(); plt.show()

---

### Exercise 4 — Research Thinking (Open-Ended)

Consider a PCR (Polymerase Chain Reaction) protocol that a biologist would perform:

1. Label tubes (pick marker, write IDs)
2. Add template DNA (pipette from stock tube)
3. Add primers (pipette, two reagents)
4. Add PCR master mix (pipette)
5. Cap all tubes
6. Load tubes into thermocycler slots
7. Set thermocycler program (turn knobs, read display)
8. Press Start, wait
9. Retrieve tubes

**Questions:**  
a) Which steps involve which biological primitives from AutoBio?  
b) Which steps would be hardest for current VLA models and why?  
c) Sketch a benchmark task graph for this protocol — which sub-tasks must be sequential vs. parallelisable?  
d) If this is a Hard task (9+ steps), what joint SR do you expect from a model with 75% per-step SSR?

In [ ]:
# ── Exercise 4 — Guided Answer ────────────────────────────────────────────────

# (a) Primitive mapping
pcr_steps = [
    ('Label tubes',          ['pick_tube', 'press_button']),   # simplified
    ('Add template DNA',     ['open_cap', 'pipette', 'close_cap']),
    ('Add primers',          ['open_cap', 'pipette', 'pipette', 'close_cap']),
    ('Add master mix',       ['open_cap', 'pipette', 'close_cap']),
    ('Cap all tubes',        ['open_cap', 'close_cap'] * 8),   # 8 tubes
    ('Load thermocycler',    ['pick_tube', 'insert_slot'] * 8),
    ('Set program',          ['turn_knob', 'turn_knob', 'read_display']),
    ('Start',                ['press_button']),
    ('Retrieve tubes',       ['pick_tube'] * 8),
]

print("(a) Primitive mapping for PCR protocol:")
print("-" * 60)
total_subtasks = 0
for step, prims in pcr_steps:
    print(f"  {step:25s}: {', '.join(prims)}")
    total_subtasks += len(prims)
print(f"\nTotal estimated sub-tasks: {total_subtasks}")

# (b) Hardest steps
print("\n(b) Hardest steps for current VLA models:")
hard = [
    ("pipette", "Sub-mm tip alignment into small tube openings"),
    ("read_display", "Reading temperature/time from LCD display"),
    ("insert_slot", "Precise slot alignment × 8 tubes"),
    ("turn_knob", "Counting exact number of detent clicks"),
]
for name, reason in hard:
    print(f"  - {name:15s}: {reason}")

# (d) Compounding error
p_step = 0.75
n_steps = total_subtasks
joint_sr = p_step ** n_steps
print(f"\n(d) With p=0.75 per step and {n_steps} total sub-tasks:")
print(f"    Joint SR = 0.75^{n_steps} = {joint_sr:.2e} ≈ {joint_sr*100:.4f}%")
print("    → Essentially 0. This is why high-precision sub-policies and error recovery matter!")

---
## 🎉 Congratulations — You've Completed the AutoBio Tutorial!

### What You've Learned

- **Why** biology lab automation is uniquely challenging (precision, transparency, digital UI, long horizon)
- **How** AutoBio's three-layer framework (Assets → Physics → Rendering) works
- **The math** behind thread helix, detent potential, eccentric orbit, and Fresnel rendering
- **How to evaluate** VLA models using SR and SSR metrics
- **Why** state-of-the-art models still fail badly on lab tasks
- **The compounding error** problem in long-horizon robotics

### Next Steps

1. 🔗 **Explore the repository:** https://github.com/autobio-bench/AutoBio
2. 📄 **Read the full paper:** arXiv:2505.14030
3. 🤖 **Run the simulator:** Install AutoBio, try loading a centrifuge task
4. 🧪 **Fine-tune a VLA model:** Use the provided demonstration data with OpenVLA
5. 💡 **Propose improvements:** Can you design a new physics plugin for **centrifugation dynamics** (sedimentation)?